# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/peachy-naomi/flyrank_ml_tasks/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


One row means: One unique Page-Query pair (content_hash_id + query_hash_id).
Time Window: March 2026.

The Grain: One row represents one unique content item (content_hash_id) per day.
The Window: I am auditing the month of March 2026 (2026-03).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Features: avg_position_prev30, clicks_prev30, impressions_prev30, query_char_count.
Label (Proxy): is_declining (Calculated as clicks_last30 < clicks_prev30).
Excluded: clicks_last30.
Why? This is the Leakage Trap. You cannot use the clicks that happened at the end of the month to predict a decline at the start of the month.

Features: gsc_avg_position, gsc_impressions, ga4_sessions, and ctr (clicks/impressions).
Label (Proxy): is_declining (derived from trend_direction == 'down').
Context: client_hash_id, report_date, month.
Excluded: trend_pct.
Why Excluded? It is a "leakage" field. It contains the exact percentage of decline calculated after the month ended. Using it would be "cheating" because it wouldn't be known at the moment we make the decision

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
import duckdb
import os
from google.colab import userdata
from huggingface_hub import hf_hub_download

# 1. Setup
token = userdata.get('HF_TOKEN')
con = duckdb.connect()

# 2. Path to the 90d Query Table (We already downloaded this)
# If you restarted, this will re-download it.
path_90d = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_query_90d.parquet",
    repo_type="dataset",
    token=token
)

# --- FACT 1: THE GRAIN ---
print("\n--- Fact 1: Verifying Grain (Content-Query Pairs) ---")
query1 = f"SELECT count(*) as total_rows, count(distinct content_hash_id || query_hash_id) as unique_pairs FROM '{path_90d}'"
display(con.execute(query1).df())

# --- FACT 2: DATA VOLUME ---
print("\n--- Fact 2: Row Counts ---")
query2 = f"SELECT count(*) as row_count FROM '{path_90d}'"
display(con.execute(query2).df())

# --- FACT 3: AVAILABILITY (Using clicks_prev30) ---
print("\n--- Fact 3: Availability (Rows with traffic to analyze) ---")
query3 = f"SELECT count(*) as total, count(CASE WHEN clicks_prev30 > 0 THEN 1 END) as rows_with_clicks FROM '{path_90d}'"
display(con.execute(query3).df())

# --- THE LEAKAGE TRAP EXPERIMENT ---
print("\n--- Step 4: The Leakage Trap ---")
# We calculate the label (is_declining) and include the "Trap" (clicks_last30)
query_trap = f"""
SELECT
    avg_position_prev30,
    clicks_prev30,
    query_char_count,
    (clicks_last30 < clicks_prev30)::int as is_declining,
    clicks_last30 as LEAKAGE_TRAP
FROM '{path_90d}'
WHERE clicks_prev30 > 5
LIMIT 10000
"""
df_final = con.execute(query_trap).df()

# Show the Trap
correlation = df_final['LEAKAGE_TRAP'].corr(df_final['is_declining'])
print(f"Leakage Trap Correlation (Future Clicks vs. Decline): {correlation:.4f}")

# Clean up
df_final = df_final.drop(columns=['LEAKAGE_TRAP'])
print("\nTrap Sprung: Future data removed. Final honest feature frame ready.")
display(df_final.head())


--- Fact 1: Verifying Grain (Content-Query Pairs) ---


,total_rows,unique_pairs
0,2414248,2414248



--- Fact 2: Row Counts ---


,row_count
0,2414248



--- Fact 3: Availability (Rows with traffic to analyze) ---


,total,rows_with_clicks
0,2414248,95826



--- Step 4: The Leakage Trap ---
Leakage Trap Correlation (Future Clicks vs. Decline): -0.3309

Trap Sprung: Future data removed. Final honest feature frame ready.


,avg_position_prev30,clicks_prev30,query_char_count,is_declining
0,1.077778,14,42,1
1,2.382379,8,23,1
2,1.182979,6,37,1
3,10.331647,6,37,1
4,0.971800,7,31,0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Threshold Bias: I am defining "decline" as any drop in clicks. However, small drops (e.g., from 10 clicks to 9) might just be random noise, not true content decay.
New Pages: Pages that didn't exist 60 days ago will have 0 for clicks_prev30, making it impossible to calculate a trend for them

Daily Noise: Since the grain is daily, traffic spikes on a Tuesday vs. a Sunday might look like "trends" to a weak model.
External Context: It doesn't know if a competitor launched a massive ad campaign or if it was a national holiday.
Seasonality: A decline in "Winter Coats" in March is normal, but the data might flag it as "content decay" because it doesn't have a "Seasonality" feature yet.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.